In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [5]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.")],
        "email": "سلام سینا، فردا برای جلسه‌مان دیر می‌رسم. می‌توانیم وقت دیگری بگذاریم؟ با احترام، علیرضا."
    },
    config=config
)

In [15]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'سلام '
                                                                          'علیرضا، '
                                                                          'اشکالی '
                                                                          'نیست. '
                                                                          'می\u200cتونیم '
                                                                          'زمان '
                                                                          'جلسه '
                                                                          'را '
                                                                          'تغییر '
                                                                          'بدیم. '
                                                                          'لطفاً '
                                                                          'زمان '


In [17]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'سلام علیرضا، اشکالی نیست. می\u200cتونیم زمان جلسه را تغییر بدیم. لطفاً زمان مناسبت را بفرمایید. اگر دوست دارید، من دو گزینه هم پیشنهاد می\u200cکنم: فردا ساعت ۱۶:۰۰ یا پس\u200cفردا ساعت ۱۰:۰۰. کدام\u200cیک برای شما مناسب است؟ با احترام، سینا'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'سلام علیرضا، اشکالی نیست. می\\u200cتونیم زمان جلسه را تغییر بدیم. لطفاً زمان مناسبت را بفرمایید. اگر دوست دارید، من دو گزینه هم پیشنهاد می\\u200cکنم: فردا ساعت ۱۶:۰۰ یا پس\\u200cفردا ساعت ۱۰:۰۰. کدام\\u200cیک برای شما مناسب است؟ با احترام، سینا'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='804c03e28654785f30165763eaaead5f')]


In [19]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

سلام علیرضا، اشکالی نیست. می‌تونیم زمان جلسه را تغییر بدیم. لطفاً زمان مناسبت را بفرمایید. اگر دوست دارید، من دو گزینه هم پیشنهاد می‌کنم: فردا ساعت ۱۶:۰۰ یا پس‌فردا ساعت ۱۰:۰۰. کدام‌یک برای شما مناسب است؟ با احترام، سینا


## تأیید

In [21]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='77b41025-7078-4f63-b74d-2ceb2c752178'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 339, 'prompt_tokens': 166, 'total_tokens': 505, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqbxI0YaeXsnVZJdnuTepbzyIUs5v', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec594-e247-7b82-95e7-c658d42e2128-0', tool_calls=[{'name': 'read_email', 'args':

## رد کردن

In [23]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "نه، لطفاً امضا کن - رهبر مهربانت، سین."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='77b41025-7078-4f63-b74d-2ceb2c752178'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 339, 'prompt_tokens': 166, 'total_tokens': 505, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqbxI0YaeXsnVZJdnuTepbzyIUs5v', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec594-e247-7b82-95e7-c658d42e2128-0', tool_calls=[{'name': 'read_email', 'args':

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## ویرایش

In [25]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "این دیگر از حد گذشت، اخراجی!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='77b41025-7078-4f63-b74d-2ceb2c752178'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 339, 'prompt_tokens': 166, 'total_tokens': 505, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqbxI0YaeXsnVZJdnuTepbzyIUs5v', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec594-e247-7b82-95e7-c658d42e2128-0', tool_calls=[{'name': 'read_email', 'args':